<a href="https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rule: Flag a content item for refresh review if it is stale (in the oldest 25% of content
by days since last update — about 104+ days for this dataset), still visible in search
(impressions_90d >= 500), and underperforming — either low CTR relative to its position,
or a weak average position despite visibility.

Reason codes:
- stale_but_visible: stale + visible, no strong CTR/position problem yet
- visible_low_ctr: visible, CTR below the median for its position bucket
- slipping_position: avg_position worse than 20, despite meaningful impressions
- low_priority: fails the stale/visible gate, score = 0

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
print(list(df.columns))


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

stale_threshold = df["days_since_last_update"].quantile(0.75)
stale   = df["days_since_last_update"] >= stale_threshold
visible = df["impressions_90d"] >= 500

df["position_bucket"] = pd.qcut(df["avg_position"].replace(0, np.nan), 10, duplicates="drop")
median_ctr_by_bucket = df.groupby("position_bucket", observed=True)["ctr"].transform("median")
low_ctr = df["ctr"] < median_ctr_by_bucket

slipping = df["avg_position"] > 20

df["score"] = (stale & visible).astype(int) * df["impressions_90d"] * (1 + low_ctr.astype(int) + slipping.astype(int))

def reason(row_stale, row_visible, row_low_ctr, row_slipping):
    if not (row_stale and row_visible):
        return "low_priority"
    if row_slipping:
        return "slipping_position"
    if row_low_ctr:
        return "visible_low_ctr"
    return "stale_but_visible"

df["reason_code"] = [reason(s, v, c, p) for s, v, c, p in zip(stale, visible, low_ctr, slipping)]

df_ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

label_col = "is_declining_label" if "is_declining_label" in df_ranked.columns else None
if label_col:
    def precision_at_k(labels, k):
        return labels.iloc[:k].mean()
    base_rate = df_ranked[label_col].mean()
    for k in [20, 50, 100]:
        p_at_k = precision_at_k(df_ranked[label_col], k)
        print(f"precision@{k}: {p_at_k:.3f}  |  base rate: {base_rate:.3f}")
else:
    print("No label column found — skipping precision@K.")

Path("work/outputs").mkdir(parents=True, exist_ok=True)
df_ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Stale threshold: {stale_threshold:.0f} days")
print(f"Wrote {len(df_ranked)} rows. Non-zero scores: {(df_ranked['score'] > 0).sum()}")
print(df_ranked['reason_code'].value_counts())

No label column found — skipping precision@K.
Stale threshold: 104 days
Wrote 30000 rows. Non-zero scores: 6542
reason_code
low_priority         23458
stale_but_visible     2819
slipping_position     2204
visible_low_ctr       1519
Name: count, dtype: int64


In [16]:
print("stale count:", stale.sum())
print("visible count:", visible.sum())
print("both (AND):", (stale & visible).sum())

stale count: 9091
visible count: 16726
both (AND): 6542


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review:

#1 content_5fe46e04994d | visible_low_ctr | conf: HIGH — pos 4.2 with 517K impressions but
CTR of only 0.14% is far below what position 4 normally earns (often 5-10%). Strong refresh
candidate — likely a title/meta mismatch or intent problem, not a ranking problem.
Wrong if: this traffic is branded/navigational and users already know the destination
(low CTR expected regardless of snippet quality).

#2 content_2dba2b1f9536 | slipping_position | conf: MEDIUM — pos 27.9, but 443K impressions
is huge for page 3. Possibly an average across many long-tail queries, some page-1, some not.
Wrong if: avg_position here is a blended average hiding a few strong queries — a real refresh
target might be narrower than "the whole page."

#3 content_b28d1efd668f | slipping_position | conf: LOW — pos 26.2, CTR 0.06%. Both numbers
are roughly what you'd expect at that position anyway, so the rule may be flagging normal
page-3 behavior, not a real problem.
Wrong if: this is simply

In [19]:
top20 = df_ranked.head(20)[["content_id", "score", "reason_code", "days_since_last_update", "impressions_90d", "ctr", "avg_position"]]

for i, row in top20.iterrows():
    print(f"#{i+1} | content_id={row['content_id']} | reason={row['reason_code']} | score={row['score']:.0f}")
    print(f"    stale={row['days_since_last_update']}d, impressions_90d={row['impressions_90d']}, ctr={row['ctr']}, avg_position={row['avg_position']}")
    print()

top20.to_csv("work/outputs/baseline_top20_review.csv", index=False)

#1 | content_id=content_5fe46e04994d | reason=visible_low_ctr | score=1035430
    stale=104d, impressions_90d=517715, ctr=0.14, avg_position=4.2

#2 | content_id=content_2dba2b1f9536 | reason=slipping_position | score=886868
    stale=104d, impressions_90d=443434, ctr=0.21, avg_position=27.9

#3 | content_id=content_b28d1efd668f | reason=slipping_position | score=859824
    stale=104d, impressions_90d=286608, ctr=0.06, avg_position=26.2

#4 | content_id=content_813e88069237 | reason=slipping_position | score=700683
    stale=104d, impressions_90d=233561, ctr=0.06, avg_position=26.2

#5 | content_id=content_cb112fce36be | reason=visible_low_ctr | score=619820
    stale=104d, impressions_90d=309910, ctr=0.16, avg_position=5.6

#6 | content_id=content_36ff89c8214e | reason=visible_low_ctr | score=590194
    stale=104d, impressions_90d=295097, ctr=0.05, avg_position=7.3

#7 | content_id=content_8b36799b7e44 | reason=slipping_position | score=424200
    stale=104d, impressions_90d=141400, c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# --- 1. Leakage check: confirm forbidden/unused columns never entered the score ---
FEATURES_USED = {"days_since_last_update", "impressions_90d", "ctr", "avg_position"}
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label"}

leaked = FEATURES_USED & FORBIDDEN
print("Leakage check:", "FAIL — forbidden columns used:" if leaked else "PASS — no forbidden columns in FEATURES_USED", leaked if leaked else "")

# double-check no _prev_30d / _last_30d columns snuck into scoring logic
future_window_cols = [c for c in df.columns if "prev_30d" in c or "last_30d" in c]
print("Trailing/forward window columns available but unused in score:", future_window_cols)

# --- 2. Programmatically flag weak picks in the top 20 ---
# A "weak pick" = flagged as needing review, but CTR is already AT or ABOVE
# the position-bucket median (i.e. nothing looks broken relative to peers at that rank)
top20 = df_ranked.head(20).copy()
top20["ctr_vs_bucket_median"] = top20["ctr"] - median_ctr_by_bucket.loc[top20.index]
top20["weak_pick"] = top20["ctr_vs_bucket_median"] >= 0

weak = top20[top20["weak_pick"]][["content_id", "reason_code", "ctr", "avg_position", "ctr_vs_bucket_median"]]
print(f"\n{len(weak)} weak picks found in top 20 (CTR at/above position-bucket median despite being flagged):")
print(weak.to_string(index=False))

Leakage check: PASS — no forbidden columns in FEATURES_USED 
Trailing/forward window columns available but unused in score: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

9 weak picks found in top 20 (CTR at/above position-bucket median despite being flagged):
          content_id       reason_code  ctr  avg_position  ctr_vs_bucket_median
content_5fe46e04994d   visible_low_ctr 0.14           4.2                  0.03
content_2dba2b1f9536 slipping_position 0.21          27.9                  0.13
content_b28d1efd668f slipping_position 0.06          26.2                  0.02
content_cb112fce36be   visible_low_ctr 0.16           5.6                  0.16
content_b511d4bc4ad2 slipping_position 0.14          27.9                  0.14
content_05e9b4cd9ccf slipping_position 0.08          22.1                  0.08
content_2c2606c5d176 stale_but_visible 0.53           4.2                  0.49
content_9532f197

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.